# SceneTwin — Description Gain on Real VideoA11y/VATEX Clips

This notebook runs the decisive SceneTwin experiment on the 20 real VATEX-overlap clips:

```text
P_AV = TRIBE(original audiovisual video)
P_A  = TRIBE(audio-only soundtrack)
P_D  = TRIBE(candidate description text)

DescriptionGain = cos(P_AV, P_D) - cos(P_AV, P_A)
```

The test is whether VideoA11y AD descriptions recover more predicted scene signal than generic same-scene VATEX captions:

```text
DG(tier3_va11y) > DG(tier2_vatex_long) > DG(tier1_vatex_short) > DG(tier0_cross)
```

This is checkpointed per clip and per tier so an interrupted Colab runtime can resume without losing completed TRIBE runs.


## 1. Runtime

Use a GPU runtime:

- `Runtime` -> `Change runtime type`
- `Hardware accelerator` -> `GPU`

Run the install cell once. It intentionally restarts the runtime after installing TRIBE and pinned dependencies.


In [ ]:
!rm -rf /content/tribev2
!git clone https://github.com/facebookresearch/tribev2.git
!pip install -e tribev2 --quiet
!pip install huggingface_hub moviepy pandas scipy matplotlib seaborn "numpy==2.2.6" --quiet

# Restart runtime so numpy / torch extensions load cleanly.
import os
os.kill(os.getpid(), 9)


## 2. Upload Data Bundle

Before running this notebook, create a zip locally with:

```bash
cd /Users/adarsha/njbda
zip -r /Users/adarsha/Knowledge/output/scenetwin_description_gain_bundle.zip vatex_eval_clips.json vatex_clips
```

Upload that zip in the next cell. The extracted layout should be:

```text
/content/scenetwin_real_eval/vatex_eval_clips.json
/content/scenetwin_real_eval/vatex_clips/clip_00.mp4
/content/scenetwin_real_eval/vatex_clips/clip_01.mp4
...
```


In [ ]:
from pathlib import Path
from google.colab import files
import zipfile

ROOT = Path('/content/scenetwin_real_eval')
ROOT.mkdir(parents=True, exist_ok=True)

if not (ROOT / 'vatex_eval_clips.json').exists():
    print('Upload scenetwin_description_gain_bundle.zip')
    uploaded = files.upload()
    zip_name = next(iter(uploaded.keys()))
    with zipfile.ZipFile(zip_name) as zf:
        zf.extractall(ROOT)
else:
    print('Bundle already extracted; reusing existing files.')

print('Files under ROOT:')
for p in sorted(ROOT.iterdir()):
    print(' ', p)


## 3. Load TRIBE and Helpers

This cell logs into Hugging Face, loads TRIBE once, and defines helpers for event construction, audio extraction, tensor alignment, and scoring.


In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, kendalltau
try:
    from moviepy.editor import VideoFileClip
except Exception:
    from moviepy import VideoFileClip
import matplotlib.pyplot as plt
import seaborn as sns

from huggingface_hub import login
login()

sys.path.insert(0, '/content/tribev2')
from tribev2 import TribeModel

ROOT = Path('/content/scenetwin_real_eval')
CLIPS_JSON = ROOT / 'vatex_eval_clips.json'
CLIPS_DIR = ROOT / 'vatex_clips'
OUT_DIR = Path('/content/scenetwin_description_gain')
PRED_DIR = OUT_DIR / 'preds'
TEXT_DIR = OUT_DIR / 'texts'
AUDIO_DIR = OUT_DIR / 'audio'
for d in [OUT_DIR, PRED_DIR, TEXT_DIR, AUDIO_DIR]:
    d.mkdir(parents=True, exist_ok=True)

with CLIPS_JSON.open() as f:
    clips = json.load(f)

TIER_KEYS = ['tier3_va11y', 'tier2_vatex_long', 'tier1_vatex_short', 'tier0_cross']
GT = {'tier3_va11y': 3, 'tier2_vatex_long': 2, 'tier1_vatex_short': 1, 'tier0_cross': 0}

def find_clip_path(idx):
    matches = sorted(CLIPS_DIR.glob(f'clip_{idx:02d}.*'))
    if not matches:
        return None
    return matches[0]

def extract_audio(video_path, idx):
    audio_path = AUDIO_DIR / f'clip_{idx:02d}.wav'
    if audio_path.exists():
        return audio_path
    clip = VideoFileClip(str(video_path))
    if clip.audio is None:
        raise ValueError(f'No audio track in {video_path}')
    try:
        clip.audio.write_audiofile(str(audio_path), fps=16000, verbose=False, logger=None)
    except TypeError:
        clip.audio.write_audiofile(str(audio_path), fps=16000, logger=None)
    clip.close()
    return audio_path

def save_text(idx, tier, text):
    path = TEXT_DIR / f'clip_{idx:02d}_{tier}.txt'
    path.write_text(text, encoding='utf-8')
    return path

def mean_cosine(a, b):
    a = np.asarray(a)
    b = np.asarray(b)
    if a.ndim == 2:
        a = a.mean(axis=0)
    if b.ndim == 2:
        b = b.mean(axis=0)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    if denom == 0:
        return float('nan')
    return float(np.dot(a, b) / denom)

def predict_cached(model, name, events_fn):
    path = PRED_DIR / f'{name}.npy'
    if path.exists():
        return np.load(path)
    print(f'Running TRIBE: {name}')
    events = events_fn()
    preds, _ = model.predict(events)
    np.save(path, preds)
    print(f'  saved {path.name} {preds.shape}')
    return preds

print(f'Loaded {len(clips)} clip metadata rows.')
valid = [(i, c, find_clip_path(i)) for i, c in enumerate(clips) if find_clip_path(i) is not None]
print(f'Found {len(valid)} local clip files.')

print('Loading TRIBE v2...')
model = TribeModel.from_pretrained('facebook/tribev2', cache_folder='/content/tribe_cache')
print('Model loaded.')


## 4. Run Description Gain

This is the long-running cell. Expect roughly 2-4 hours on Colab T4 for 20 clips depending on caching and transcription/TTS latency.


In [ ]:
rows = []

for idx, clip_meta, video_path in valid:
    print(f'\n=== clip_{idx:02d}: {clip_meta.get("video_id", "unknown")} ===')
    p_av = predict_cached(
        model,
        f'clip_{idx:02d}_P_AV',
        lambda video_path=video_path: model.get_events_dataframe(video_path=str(video_path)),
    )

    try:
        audio_path = extract_audio(video_path, idx)
        p_a = predict_cached(
            model,
            f'clip_{idx:02d}_P_A',
            lambda audio_path=audio_path: model.get_events_dataframe(audio_path=str(audio_path)),
        )
        av_audio_cos = mean_cosine(p_av, p_a)
    except Exception as exc:
        print(f'  audio-only failed: {exc}')
        p_a = None
        av_audio_cos = float('nan')

    for tier in TIER_KEYS:
        text_path = save_text(idx, tier, clip_meta[tier])
        p_d = predict_cached(
            model,
            f'clip_{idx:02d}_{tier}_P_D',
            lambda text_path=text_path: model.get_events_dataframe(text_path=str(text_path)),
        )
        av_desc_cos = mean_cosine(p_av, p_d)
        dg = av_desc_cos - av_audio_cos
        rows.append({
            'clip_idx': idx,
            'video_id': clip_meta.get('video_id'),
            'category': clip_meta.get('category'),
            'tier': tier,
            'gt': GT[tier],
            'av_desc_cos': av_desc_cos,
            'av_audio_cos': av_audio_cos,
            'description_gain': dg,
            'desc_words': len(clip_meta[tier].split()),
            'video_path': str(video_path),
        })
        pd.DataFrame(rows).to_csv(OUT_DIR / 'description_gain_partial.csv', index=False)
        print(f'  {tier}: cos(AV,D)={av_desc_cos:.4f}  cos(AV,A)={av_audio_cos:.4f}  DG={dg:.4f}')

df = pd.DataFrame(rows)
df.to_csv(OUT_DIR / 'description_gain_results.csv', index=False)
df.head()


## 5. Analyze and Export Results


In [ ]:
df = pd.read_csv(OUT_DIR / 'description_gain_results.csv')

rho, p_rho = spearmanr(df['gt'], df['description_gain'], nan_policy='omit')
tau, p_tau = kendalltau(df['gt'], df['description_gain'], nan_policy='omit')
print(f'Spearman rho={rho:.4f}, p={p_rho:.4g}')
print(f'Kendall tau={tau:.4f}, p={p_tau:.4g}')

means = df.groupby('tier').agg(
    description_gain=('description_gain', 'mean'),
    av_desc_cos=('av_desc_cos', 'mean'),
    av_audio_cos=('av_audio_cos', 'mean'),
    desc_words=('desc_words', 'mean'),
).loc[TIER_KEYS]
display(means)

pair_rows = []
for comp in ['tier2_vatex_long', 'tier1_vatex_short', 'tier0_cross']:
    wins = 0
    total = 0
    for clip_idx, group in df.groupby('clip_idx'):
        t3 = group[group['tier'] == 'tier3_va11y']['description_gain']
        tx = group[group['tier'] == comp]['description_gain']
        if len(t3) and len(tx):
            total += 1
            wins += int(float(t3.iloc[0]) > float(tx.iloc[0]))
    pair_rows.append({'comparison': f'tier3_va11y > {comp}', 'wins': wins, 'total': total, 'accuracy': wins / total if total else np.nan})
pairwise = pd.DataFrame(pair_rows)
display(pairwise)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=df, x='tier', y='description_gain', order=TIER_KEYS, errorbar='se')
plt.xticks(rotation=20, ha='right')
plt.title('SceneTwin Description Gain by Description Tier')
plt.ylabel('Description Gain = cos(AV,D) - cos(AV,A)')
plt.tight_layout()
plt.savefig(OUT_DIR / 'description_gain_tiers.png', dpi=180)
plt.show()

report = f'''---
title: "SceneTwin Description Gain — Real VideoA11y/VATEX Clips"
category: research
tags: [SceneTwin, TRIBE, Description-Gain, VideoA11y, VATEX]
created: 2026-05-02
updated: 2026-05-02
---

# SceneTwin Description Gain — Real VideoA11y/VATEX Clips

## Metric

`DescriptionGain = cos(P_AV, P_D) - cos(P_AV, P_A)`

- `P_AV`: TRIBE prediction for original audiovisual video
- `P_A`: TRIBE prediction for audio-only soundtrack
- `P_D`: TRIBE prediction for candidate text description

## Main Result

| Metric | Value |
|---|---:|
| Spearman rho | {rho:.4f} |
| Spearman p | {p_rho:.4g} |
| Kendall tau | {tau:.4f} |
| Kendall p | {p_tau:.4g} |

## Mean Scores by Tier

{means.to_markdown()}

## Pairwise Accuracy

{pairwise.to_markdown(index=False)}

## Files

- `description_gain_results.csv`
- `description_gain_partial.csv`
- `description_gain_tiers.png`
- `preds/*.npy`
'''
(OUT_DIR / 'scenetwin-description-gain-report.md').write_text(report, encoding='utf-8')
print(report)


In [ ]:
from google.colab import files

!cd /content && zip -qr scenetwin_description_gain_outputs.zip scenetwin_description_gain
files.download('/content/scenetwin_description_gain_outputs.zip')
